In [ ]:
# READING GUIDE
# Define basic AR(p) utilities used throughout the notebook.
# The companion matrix converts the AR recurrence into a first-order vector recursion.
# Its spectral radius is below one exactly when the chosen AR coefficients are stationary.
# Partial autocorrelations are converted recursively into a guaranteed-stationary coefficient vector.
# The recurrence vector q=(1,-phi_1,...,-phi_p) is normal to the AR innovation slab.

# FUNCTION: companion_spectral_radius
# Build the AR companion matrix and return its largest eigenvalue magnitude.
# A value below 1 is the numerical stationarity check used everywhere below.
def companion_spectral_radius(phi):
    phi = np.asarray(phi, dtype=float)
    matrix = np.zeros((len(phi), len(phi)))
    # The first row applies the AR recurrence coefficients.
    matrix[0] = phi
    if len(phi) > 1:
        # Lower rows shift each lagged state down by one position.
        matrix[1:, :-1] = np.eye(len(phi) - 1)
    return float(np.max(np.abs(np.linalg.eigvals(matrix))))


# FUNCTION: automatic_stationary_ar_coefficients
# Create a reproducible full AR(p) coefficient vector from decaying PACF values.
# The Levinson recursion preserves stationarity when every PACF lies in (-1,1).
def automatic_stationary_ar_coefficients(order, first_pacf=0.95, decay=-0.55):
    """Generate stationary AR coefficients from decaying partial autocorrelations."""
    if not isinstance(order, (int, np.integer)) or order < 1:
        raise ValueError("order must be a positive integer")
    if not (abs(first_pacf) < 1 and abs(decay) < 1):
        raise ValueError("first_pacf and decay must have absolute value below one")

    # Create the PACF sequence kappa_j used by the Levinson recursion.
    pacf = first_pacf * decay ** np.arange(order)
    phi = np.array([pacf[0]])
    for m in range(2, order + 1):
        # Keep the previous recursion level because the update uses it in both orders.
        old_phi = phi.copy()
        reflection = pacf[m - 1]
        phi = np.empty(m)
        # Update earlier AR coefficients using the new reflection coefficient.
        phi[:-1] = old_phi - reflection * old_phi[::-1]
        # The newest PACF becomes the highest-lag AR coefficient.
        phi[-1] = reflection
    return phi


# FUNCTION: check_stationarity
# Validate a coefficient vector once and fail early for a nonstationary model.
def check_stationarity(phi):
    radius = companion_spectral_radius(phi)
    if radius >= 1:
        raise ValueError(f"AR coefficients are not stationary: spectral radius = {radius:.4f}")
    return radius


# FUNCTION: ar_recurrence_vector
# Return q_phi=(1,-phi_1,...,-phi_p), the normal vector of an AR innovation slab.
def ar_recurrence_vector(phi):
    return np.r_[1.0, -np.asarray(phi, dtype=float)]


# FUNCTION: ar_recurrence_direction
# Normalize q_phi so widths can be compared on the unit sphere.
def ar_recurrence_direction(phi):
    raw = ar_recurrence_vector(phi)
    return raw / np.linalg.norm(raw)


# FUNCTION: ar_recurrence_directions
# Embed q_phi at every possible coordinate offset inside R^k.
# No such direction fits when k<p+1, so that case returns an empty list.
def ar_recurrence_directions(phi, embedding_dimension):
    """All coordinate-shifted thesis-convention AR recurrence directions in R^k."""
    raw = ar_recurrence_vector(phi)
    p_plus_one = len(raw)
    k = int(embedding_dimension)
    if k < p_plus_one:
        return []

    directions = []
    for start in range(k - p_plus_one + 1):
        direction = np.zeros(k)
        direction[start:start + p_plus_one] = raw
        directions.append(direction / np.linalg.norm(direction))
    return directions
